In [1]:
!pip install --quiet torch chromadb sentence-transformers numpy bertopic packaging==24.1 umap-learn scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.18 requires numpy<2,>=1.26.4; python_version < "3.12", but you have numpy 1.23.5 which is incompatible.


In [2]:
import chromadb
from umap import UMAP
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from nltk.corpus import stopwords as nltk_stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from collections import defaultdict
import json
import os

In [3]:
# Verbindung zur bestehenden ChromaDB
chroma_client = chromadb.PersistentClient(path="./chromadb")
wahlprogramme_collection = chroma_client.get_collection(name="wahlprogramme")
plenarsitzungen_collection = chroma_client.get_collection(name="plenarsitzungen")

In [11]:
result = plenarsitzungen_collection.get(
    where={"party": {
    "$nin": ["n/a", "Fraktionslos", "SPDCDU/CSU"]}},
    include=["documents", "metadatas", "embeddings"])

party_docs = defaultdict(list)
for doc, meta, embedding in zip(result["documents"], result["metadatas"], result["embeddings"]):
    party = meta.get("party")

    party_docs[party].append({
        "text": doc,
        "metadata": meta,
        "embedding": embedding
    })

for party, docs in party_docs.items():
    print(f"Party: {party} has {len(docs)} documents.")

Party: CDU/CSU has 31423 documents.
Party: SPD has 29704 documents.
Party: AfD has 13790 documents.
Party: BÜNDNIS 90/DIE GRÜNEN has 18204 documents.
Party: FDP has 15052 documents.
Party: DIE LINKE has 7221 documents.
Party: BSW has 478 documents.


In [12]:
parteien = ["cdu", "csu", "spd", "afd", "bündnis", "bündnisses", "90", "grünen", "fdp", "linke", "linken", "bsw"]
foermlichkeiten = ["verehrt", "verehrte", "liebe", "dr", "herr", "herren", "dame", "damen", "dank", "danke", "danken", "bitte", "bitten", "aufmerksamkeit", "beifall", "herzlich", "herzlichen", "zuruf", "werte",
                   "kollegen", "kolleginnen", "gerne", "zustimmen", "vorschläge", "unterstützen", "zuzustimmen", "zusammenarbeit", "abgeordneten", "abgeordnete", "zustimmung", "beschließen"]
stopwords_plenar = ""
with open("./german_stopwords/german_stopwords_plain.txt") as file:
    stopwords_plenar = [line.strip() for line in file if not line.startswith(";") and line.strip()]

stopwords_plenar.extend(foermlichkeiten) # entfernen der Förmlichkeitsfloskeln erhöht Qualität der Topics drastisch und sind entsprechend notwendig
stopwords_plenar.extend(parteien) # erhöht Qualität weiter, mit kleinem verlust des Kontextes

In [13]:
party_data = {}
# extrahiert Text und Embeddings für die jeweiligen Parteien
for party, docs in party_docs.items():
    texts = [doc["text"] for doc in docs]
    embeddings = np.array([doc["embedding"] for doc in docs])
    party_data[party] = {"texts": texts, "embeddings": embeddings}

In [14]:
party_models_plenar = {}

# erstellt ein BERTopic Modell pro Partei
for party, data in party_data.items():
    
    representation_model=KeyBERTInspired()  
    embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")
    vectorizer_model = CountVectorizer(stop_words=stopwords_plenar, ngram_range=(1,1), min_df=1)
    umap_model = UMAP(low_memory=False, metric='cosine', min_dist=0.0, n_neighbors=15, n_components=5, random_state=42) # setzt Generierung der Topics fest
    topic_model = BERTopic(language="german",
                           vectorizer_model=vectorizer_model, 
                           umap_model=umap_model,
                           embedding_model=embedding_model,                           
                           representation_model=representation_model,
                           top_n_words=10, 
                          )
    topics, probs = topic_model.fit_transform(data["texts"], data["embeddings"])
    
    party_models_plenar[party] = topic_model

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [15]:
for party, model in party_models_plenar.items():
    print(f"\nParty: {party}")
    
    topic_info = model.get_topic_info()
    # -1 ist das "outlier" Topic
    filtered_topic_info = topic_info[topic_info.Topic != -1]
    
    top_topics = filtered_topic_info.head(5)
    
    for idx, row in top_topics.iterrows():
        topic_id = row["Topic"]
        topic_name = row["Name"]
        topic_words_tuples = model.get_topic(topic_id)
        
        topic_words = [word for word, _ in topic_words_tuples]
        print(f"Topic {topic_id} ({topic_name}): {', '.join(topic_words)}")



Party: CDU/CSU
Topic 0 (0_gesetzentwurf_bundesregierung_umsetzen_gemeinsam): gesetzentwurf, bundesregierung, umsetzen, gemeinsam, dringend, einbringen, nachhaltigkeit, wünsche, unterstützung, gründen
Topic 1 (1_verteidigungshaushalt_verteidigungsministerin_verteidigungsminister_wehrbeauftragten): verteidigungshaushalt, verteidigungsministerin, verteidigungsminister, wehrbeauftragten, verteidigungsausgaben, bündnisverteidigung, wehrbeauftragte, verteidigungsetat, aufrüstung, landesverteidigung
Topic 2 (2_abschiebungen_migrationspolitik_asylbewerber_migrationskrise): abschiebungen, migrationspolitik, asylbewerber, migrationskrise, innenministerin, asylverfahren, staatsbürgerschaftsrecht, herkunftsstaaten, staatsangehörigkeit, grenzkontrollen
Topic 3 (3_wohnungsbau_bauministerin_bauministerium_wohnungsmarkt): wohnungsbau, bauministerin, bauministerium, wohnungsmarkt, sozialwohnungen, baugenehmigungen, neubauförderung, baupolitik, wohnungen, bauen
Topic 4 (4_bürgergeldes_bürgergeld_arbeit

In [10]:
def bertopic_to_json(party_models, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Loop over each party's BERTopic model
    for party, model in party_models.items():
        # Get all topic information as a DataFrame
        topic_info = model.get_topic_info()

        topic_info = topic_info[topic_info.Topic != -1]

        topics_list = []
        for _, row in topic_info.iterrows():
            topic_id, *topic_words = row["Name"].split("_")

            topics_list.append({
                "id": topic_id,
                "words": list(topic_words)
            })

        output_dict = {
            "party": party,
            "topics": topics_list
        }

        # Define the file path inside the output folder
        filename = f"{party.replace('/', '_')}_topics.json"
        filepath = os.path.join(output_folder, filename)

        # Export to JSON file
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(output_dict, f, ensure_ascii=False, indent=4)

In [4]:
result = wahlprogramme_collection.get(include=["documents", "metadatas", "embeddings"])

party_wahlprogramm = defaultdict(list)
for doc, meta, embedding in zip(result["documents"], result["metadatas"], result["embeddings"]):
    source = meta.get("source")

    party_wahlprogramm[source].append({
        "text": doc,
        "metadata": meta,
        "embedding": embedding
    })

for source, docs in party_wahlprogramm.items():
    print(f"Party: {source} has {len(docs)} documents.")

Party: GRUENE_Wahlprogramm.pdf has 318 documents.
Party: LINKE_Wahlprogramm.pdf has 259 documents.
Party: SPD_Wahlprogramm.pdf has 256 documents.
Party: CDU_Wahlprogramm.pdf has 275 documents.
Party: AFD_Wahlprogramm.pdf has 335 documents.
Party: FDP_Wahlprogramm.pdf has 211 documents.
Party: BSW_Wahlprogramm.pdf has 203 documents.


In [5]:
stopwords_wahlprogramm = ""
with open("./german_stopwords/german_stopwords_plain.txt") as file:
    stopwords_wahlprogramm = [line.strip() for line in file if not line.startswith(";") and line.strip()]
stopwords_wahlprogramm.extend(["wollen", "setzen"])

In [6]:
party_data_wahlprogramm = {}
# extrahiert Text und Embeddings für die jeweiligen Parteien
for party, docs in party_wahlprogramm.items():
    texts = [doc["text"] for doc in docs]
    embeddings = np.array([doc["embedding"] for doc in docs])
    party_data_wahlprogramm[party] = {"texts": texts, "embeddings": embeddings}

In [7]:
party_models_wahlprogramm = {}

for party, data in party_data_wahlprogramm.items():
    #auch möglich mit c-tf-idf hoch frequente wörter zu reduzieren, aber ungeeignet bei geringer datenmenge
    representation_model=KeyBERTInspired()  
    embedding_model = SentenceTransformer("intfloat/multilingual-e5-large")
    vectorizer_model = CountVectorizer(stop_words=stopwords_wahlprogramm, ngram_range=(1,1)) #ngram_range beeinflusst wie viele Worte als "ein Bestandteil" von Topic genommen werden. Bei 2 sind Topics eher undursichtig, bei 3 fast ganzer Satz
    umap_model = UMAP(low_memory=False, metric='cosine', min_dist=0.0, n_components=5, random_state=42) # setzt Generierung der Topics fest
    topic_model = BERTopic(language="german",
                           vectorizer_model=vectorizer_model, 
                           umap_model=umap_model,
                           embedding_model=embedding_model,                           
                           representation_model=representation_model,
                           top_n_words=10, 
                           min_topic_size=5
                          )
    topics, probs = topic_model.fit_transform(data["texts"], data["embeddings"])
    
    party_models_wahlprogramm[party] = topic_model

In [8]:
def print_top_n_topics(n, party_models):
    for party, model in party_models.items():
        print(f"\nParty: {party}")

        topic_info = model.get_topic_info()

        filtered_topic_info = topic_info[topic_info.Topic != -1]

        top_topics = filtered_topic_info.head(n)

        for idx, row in top_topics.iterrows():
            topic_id = row["Topic"]
            topic_name = row["Name"]
            topic_words_tuples = model.get_topic(topic_id)

            topic_words = [word for word, _ in topic_words_tuples]
            print(f"Topic {topic_id} ({topic_name}): {', '.join(topic_words)}")
            
print_top_n_topics(5, party_models_wahlprogramm)



Party: GRUENE_Wahlprogramm.pdf
Topic 0 (0_klimaschutzverträge_vorsorge_grünen_investitionen): klimaschutzverträge, vorsorge, grünen, investitionen, wirtschaft, klimaschutz, stärken, gesellschaft, erhalten, verbessern
Topic 1 (1_klimaschutz_ckerhaltige_großschutzgebiete_naturschonende): klimaschutz, ckerhaltige, großschutzgebiete, naturschonende, verschmutzung, lebensqualität, lebensmittel, umwelt, sauberes, pestizideinsatz
Topic 2 (2_deradikalisierungsprogramme_polizeiarbeit_kriminalität_kriminalpolizeilichen): deradikalisierungsprogramme, polizeiarbeit, kriminalität, kriminalpolizeilichen, zusammenarbeit, finanzkriminalität, staatsanwaltschaft, bundespolizei, sicherheitsbehörden, kriminalamt
Topic 3 (3_bildungsaufbruch_schulprogramm_schulsozial_aufstiegs): bildungsaufbruch, schulprogramm, schulsozial, aufstiegs, zukunft, auszubildende, schulen, fördern, einwanderung, ausbildungsbetrieben
Topic 4 (4_menschenrechte_entwicklungspolitik_humanitäre_zusammenarbeit): menschenrechte, entwick

Die deutschen stop words von nltk reichen mit 232 Wörtern nicht ganz aus. <br>
Leicht bessere Ergebnisse werden mit der german_stopwords_plain.txt Variante des nachfolgenden GitHub Repos erzielt <br>
https://github.com/solariz/german_stopwords

stop words filtern erhöht die Qualität der Topics drastisch. <br>
german_stop_words = stopwords_from_file (GitHub, 598 Wörter) <br>

CountVectorizer <br>
https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html#countvectorizer <br>
https://maartengr.github.io/BERTopic/faq.html#how-do-i-remove-stop-words <br>
vectorizer_model = CountVectorizer(stop_words=german_stop_words) <br>

Standardmäßig verwendet BERTopic für Sprachen abseits von Englisch die mini Variante <br>
base Variante erzielt bessere Ergebnisse (mehr Parameter, größerer Vektor) <br>
https://maartengr.github.io/BERTopic/faq.html#which-embedding-model-should-i-choose <br>
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2") <br>

UMAP mit gleichen Parametern wie bei BERTopic, nur mit festgesetzten random_state (erhöht nötige Rechenzeit) <br>
https://umap-learn.readthedocs.io/en/latest/reproducibility.html <br>
umap_model = UMAP(low_memory=False, metric='cosine', min_dist=0.0, n_components=5, random_state=42) <br>

In [16]:
bertopic_to_json(party_models_plenar, "plenarsitzung_topics")

In [18]:
bertopic_to_json(party_models_wahlprogramm, "wahlprogramm_topics")